# wordlesmith demo

This notebook imports the installed `wordlesmith` package (the source of truth) and shows:

1. Scoring a guess and reading a feedback pattern
2. Auto-playing a game with each strategy
3. A small sampled benchmark
4. The committed full-run comparison table and plot

Install first with `pip install -e ".[bench]"` from the repo root. Kernel: Python 3.10+.

In [1]:
from wordlesmith import feedback, pattern_to_string, simulate, get_strategy, available_strategies

# Duplicate letters: SPEED vs ABIDE -> the second E is gray (only one E in the target).
print('SPEED vs ABIDE ->', pattern_to_string(feedback('speed', 'abide')))
print('strategies:', available_strategies())

SPEED vs ABIDE -> xxyxy
strategies: ['frequency', 'entropy', 'expected-size', 'minimax', 'random']


In [2]:
# Auto-play a target with every strategy (default pool = all valid words).
# 'maven' is a real answer missing from the original 2,315-word list.
target = 'maven'
for name in available_strategies():
    result = simulate(target, get_strategy(name))
    print(f'{name:14s} {result.turns} guesses: {result.guesses}')

frequency      5 guesses: ['sanes', 'paven', 'daven', 'haven', 'maven']
entropy        3 guesses: ['tares', 'laden', 'maven']
expected-size  4 guesses: ['lares', 'waned', 'taken', 'maven']


minimax        4 guesses: ['seria', 'laden', 'taken', 'maven']
random         4 guesses: ['varas', 'mauvy', 'mavie', 'maven']


In [3]:
# A quick sampled benchmark. curated=True uses the small 2,315-word set so this
# runs in seconds; drop it to benchmark against all valid words (slower).
from wordlesmith.benchmark import compare, format_table

results = compare(['random', 'frequency', 'entropy'], sample=200, seed=0, curated=True)
print(format_table(results))

strategy   pool     avg    median  max  fail%  time(s)
random     answers  2.360  1.0     >6   0.50   0.9    
frequency  answers  3.705  4.0     >6   0.50   1.3    
entropy    answers  3.645  4.0     6    0.00   3.6    


In [4]:
# The committed full run: every valid word. See benchmarks/results/official/.
import csv
from pathlib import Path

official = Path('..') / 'benchmarks' / 'results' / 'official' / 'primary_valid.csv'
if official.exists():
    with open(official) as f:
        for row in csv.DictReader(f):
            print(f"{row['strategy']:14s} avg={row['average']} max={row['max']} fail%={row['fail_pct']}")
else:
    print('Run scripts/run_official_benchmark.py to generate the full results.')

random         avg=5.0609 max=7 fail%=16.6813
frequency      avg=4.9225 max=7 fail%=14.5675
expected-size  avg=4.5848 max=7 fail%=10.5688
minimax        avg=4.6582 max=7 fail%=11.2891
entropy        avg=4.5234 max=7 fail%=9.4716


![guess distribution](../benchmarks/results/official/distribution_valid.png)